In [ ]:
# SPDX-License-Identifier: GPL-3.0-or-later

In [ ]:
import os
import subprocess

import numpy as np
from jupyterlab_h5web import H5Web
from pint import Quantity
from score_config import ScoreConfig
from score_config_utils import ureg

cwd_path = os.getcwd()
print(cwd_path)

### Create a configuration file for SCORE

<div class="alert alert-block alert-info">Modify those values in this populated template that match your case.</div>

This generates a NeXus/HDF5 file that matches the [`NXmicrostructure_score_config.nxdl.xml` contributed application definition](https://fairmat-nfdi.github.io/nexus_definitions/classes/contributed_definitions/NXmicrostructure_score_config.html).

In [ ]:
MY_MATERIAL: list[tuple[str, Quantity]] = [
    (
        "melting_temperature",
        ureg.Quantity(np.float64(650.0), ureg.degC).to(ureg.kelvin),
    ),
    (
        "shear_modulus_zero",
        ureg.Quantity(np.float64(26.0), ureg.gigapascal).to(ureg.pascal),
    ),
    ("nadal_lepoac_a", ureg.Quantity(np.float64(0.469))),
    ("nadal_lepoac_zeta", ureg.Quantity(np.float64(0.04))),
    ("burgers_vector_zero", ureg.Quantity(np.float64(0.286), ureg.nanometer)),
    # shown here how to consider a constant Burgers vector
    ("lattice_expansion_second", ureg.Quantity(0.0, (ureg.kelvin**-2))),
    ("lattice_expansion_first", ureg.Quantity(0.0, (ureg.kelvin**-1))),
    ("lattice_expansion_null", ureg.Quantity(0.0, ureg.nanometer)),
]

MY_USER: list[str] = [
    "Vitesh Shah",
    "Markus Kühbach",
    "Sebastian Samberger",
    "Georg Falkinger",
    "Franz Roters",
]

MY_INIT_DAMASK: dict[str, str] = {
    "file_path": f"{os.getcwd()}/../../examples/damask/Tphase_Vf_01_tensionX_remeshed_increment_19000_segmented.dream3d"
}

MY_MOBILITY_RH: list[tuple[str, Quantity]] = [
    (
        "m_null",
        ureg.Quantity(np.float64(300.0), ureg.meter**4 / (ureg.joule * ureg.second)),
    ),
    ("enthalpy", ureg.Quantity(np.float64(2.0314), ureg.eV)),
    # ("c1", ureg.Quantity(np.float64(0.9))),
    # ("c2", ureg.Quantity(np.float64(5.))),
    # ("c3", ureg.Quantity(np.float64(9.))),
]

MY_TIME_TEMPERATURE: list[tuple[str, Quantity]] = [
    (
        "time",
        ureg.Quantity(
            np.asarray(
                [
                    0.0,
                    5.8,
                    100.8,
                    103.7,
                    105.0,
                    200.8,
                    205.8,
                    208.7,
                    300.0,
                    305.8,
                    405.8,
                    408.7,
                    600.0,
                ],
                np.float64,
            ),
            ureg.second,
        ),
    ),
    (
        "temperature",
        ureg.Quantity(
            np.asarray(
                [
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                ],
                np.float64,
            ),
            ureg.degC,
        ).to(ureg.kelvin),
    ),
]

MY_GRID_CUBOIDAL: list[tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([19, 15, 11], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.5), ureg.micrometer)),
]

MY_GRID_DAMASK: list[tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([128, 52, 64], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.4982421), ureg.micrometer)),
]

In [ ]:
print("Generating a NeXus/HDF5 configuration file for examples/damask")
nxs = ScoreConfig(1)
nxs.user(MY_USER)
nxs.sample("Al")
nxs.material(MY_MATERIAL)
nxs.deformation("damask", MY_INIT_DAMASK)
nxs.nucleation("damask")
nxs.mobility("rollett_holm", MY_MOBILITY_RH)
nxs.time_temperature(MY_TIME_TEMPERATURE)
nxs.discretization(MY_GRID_DAMASK)
nxs.numerics()
nxs.get_config()
nxs.write_config_hfive_file()

In [ ]:
# optionally inspect NeXus/HDF5 file
# H5Web(f"SCORE.Config.SimID.{nxs.sim_id}.nxs")

## Run the SCORE simulation using this configuration file

This generates a NeXus/HDF5 file that matches the [`NXmicrostructure_score_results.nxdl.xml` contributed application definition](https://fairmat-nfdi.github.io/nexus_definitions/classes/contributed_definitions/NXmicrostructure_score_results.html).

In [ ]:
cmd = f"mpiexec -n 1 ./../../build/score {nxs.sim_id} {nxs.hdf5_file_name}"
with open(f"SCORE.Results.SimID.{nxs.sim_id}.nxs.log", "w") as log:
    p = subprocess.Popen(
        cmd,
        shell=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    p.wait()
if p.returncode == 0:
    print("Recrystallization simulation completed")
else:
    print(f"Pipeline failed with exit code {p.returncode}")

<div class="alert alert-info">
Note that the wrapping via subprocess causes an overhead. We recommend to move the simulations to a computer cluster.
</div>

In [ ]:
# optionally inspect the results
H5Web(f"SCORE.Results.SimID.{nxs.sim_id}.nxs")

***
This completes the tutorial.